### asyncio.Queue

异步队列用于在协程之间传递任务。

核心 API：
- `put()` / `get()`：放入 / 取出
- `task_done()`：标记「取出的这一项」已处理完
- `join()`：等待队列里所有项都被 `task_done()`

注意：`get()` 只是领取，不算完成；必须再调 `task_done()`。

Jupyter 里已有事件循环，用 `await main()`，不要用 `asyncio.run()`。

In [2]:
import asyncio


async def main():
    queue = asyncio.Queue()

    # 放入任务：未完成数变为 1
    await queue.put("文档1")

    # 领取任务，但未完成数仍然是 1
    document = await queue.get()
    print(f"开始处理：{document}")

    # 模拟处理
    await asyncio.sleep(1)
    print(f"处理完成：{document}")

    # 未完成数变为 0
    queue.task_done()

    # 因为未完成数已经是 0，所以立即通过
    await queue.join()

    print("所有任务已完成")


await main()

开始处理：文档1
处理完成：文档1
所有任务已完成


流程：

```
put("文档1")  → unfinished = 1
get()         → unfinished 仍为 1（只是领取）
处理中...
task_done()   → unfinished = 0
join()        → 立即返回
```

漏调 `task_done()`，`join()` 会一直等下去。

### 生产者 / 消费者

常见模式：一个（或多个）生产者往队列放任务，多个消费者并发处理。

In [ ]:
async def producer(queue: asyncio.Queue, items: list[str]):
    for item in items:
        await queue.put(item)
        print(f"生产：{item}")


async def consumer(name: str, queue: asyncio.Queue):
    while True:
        item = await queue.get()
        try:
            print(f"{name} 处理：{item}")
            await asyncio.sleep(0.5)
        finally:
            # 无论成功失败都要标记完成，否则 join() 永远等
            queue.task_done()


async def main():
    queue = asyncio.Queue()

    workers = [
        asyncio.create_task(consumer("消费者A", queue)),
        asyncio.create_task(consumer("消费者B", queue)),
    ]

    await producer(queue, ["任务1", "任务2", "任务3", "任务4"])

    # 等队列里所有任务都被 task_done()
    await queue.join()

    # 消费者是 while True，需要手动取消退出
    for w in workers:
        w.cancel()

    await asyncio.gather(*workers, return_exceptions=True)
    print("全部完成")


await main()

- `join()` 只保证「已 put 的项都处理完」，不会让消费者自己退出
- 消费者若是死循环，处理完后要用 `cancel()` 或哨兵值退出

### 有界队列与背压

`asyncio.Queue(maxsize=N)`：队列满时，`put()` 会挂起，直到有空位。

这就是背压：消费者跟不上时，生产者自动减速，避免内存无限堆积。

In [ ]:
async def fast_producer(queue: asyncio.Queue):
    for i in range(5):
        print(f"准备放入 {i}，当前队列大小={queue.qsize()}")
        await queue.put(i)  # 满了就在这里等待
        print(f"已放入 {i}")


async def slow_consumer(queue: asyncio.Queue):
    while True:
        item = await queue.get()
        print(f"消费 {item}")
        await asyncio.sleep(1)  # 故意慢
        queue.task_done()


async def main():
    queue = asyncio.Queue(maxsize=2)  # 最多缓存 2 个

    consumer = asyncio.create_task(slow_consumer(queue))
    await fast_producer(queue)
    await queue.join()

    consumer.cancel()
    await asyncio.gather(consumer, return_exceptions=True)


await main()
# 观察：放入 0、1 后，再 put 会被挡住，直到消费者腾出空位

### 哨兵值退出（可选）

不想用 `cancel()` 时，可以放入 `None` 作为结束信号。

In [ ]:
async def consumer(queue: asyncio.Queue):
    while True:
        item = await queue.get()
        try:
            if item is None:  # 哨兵：结束
                break
            print(f"处理 {item}")
            await asyncio.sleep(0.2)
        finally:
            queue.task_done()


async def main():
    queue = asyncio.Queue()
    worker = asyncio.create_task(consumer(queue))

    for x in ["a", "b", "c"]:
        await queue.put(x)

    await queue.put(None)  # 通知消费者退出
    await queue.join()
    await worker
    print("消费者已正常退出")


await main()

### 小结（面试常问）

| 概念 | 要点 |
|------|------|
| `Queue` | 协程间传任务，`put`/`get` 都会在需要时挂起 |
| `task_done` + `join` | `get` ≠ 完成；漏调 `task_done`，`join` 会死等 |
| 有界队列 | `maxsize` 满时 `put` 挂起 → 背压 |
| 背压作用 | 防止生产者过快撑爆内存 |
| 消费者退出 | `cancel()` 或哨兵值（如 `None`） |

和 Semaphore 的区别：
- Semaphore：限制「同时进行中的协程数」
- 有界 Queue：限制「等待处理的积压量」